In [ ]:
#quality metrics with celltype

import matplotlib.pyplot as plt
import seaborn as sns
import scanpy as sc
from matplotlib.backends.backend_pdf import PdfPages

with PdfPages("output_file.pdf") as pdf:
    tma_ids = adatas.obs['TMA_ID'].unique()
    
    for tma_id in tma_ids:
        adata = adatas[adatas.obs['TMA_ID'] == tma_id].copy()
        #qc metrics
        sc.pp.calculate_qc_metrics(adata, percent_top=(10, 20, 50, 150), inplace=True)
        cprobes = adata.obs["control_probe_counts"].sum() / adata.obs["total_counts"].sum() * 100
        cwords = adata.obs["control_codeword_counts"].sum() / adata.obs["total_counts"].sum() * 100


        fig = plt.figure(figsize=(18, 12))
        fig.suptitle(f"Quality Metrics for {tma_id}", fontsize=16, fontweight='bold')

        ax_img = fig.add_subplot(2, 1, 1)

        try:
            sc.pl.spatial(
                adata,
                color="cell_type_manual",
                spot_size=15,
                img_key="hires",
                title=f"TMA Tissue Image: {tma_id}",
                ax=ax_img,
                show=False
            )
        except Exception as e:
            ax_img.text(0.5, 0.5, f"Spatial plot not available: {e}", ha='center', va='center')
            ax_img.axis("off")

        axs = [fig.add_subplot(2, 4, i + 5) for i in range(4)]

        sns.histplot(adata.obs["total_counts"], kde=False, ax=axs[0])
        axs[0].set_title("Total transcripts")

        sns.histplot(adata.obs["n_genes_by_counts"], kde=False, ax=axs[1])
        axs[1].set_title("Unique transcripts")

        sns.histplot(adata.obs["cell_area"], kde=False, ax=axs[2])
        axs[2].set_title("Cell area")

        sns.histplot(adata.obs["nucleus_area"] / adata.obs["cell_area"], kde=False, ax=axs[3])
        axs[3].set_title("Nucleus ratio")

        plt.tight_layout(rect=[0, 0.03, 1, 0.95])
        pdf.savefig(fig)
        plt.close(fig)


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import scanpy as sc
from matplotlib.backends.backend_pdf import PdfPages

qc_data = []

tma_ids = adatas.obs['TMA_ID'].unique()
for tma_id in tma_ids:
    adata = adatas[adatas.obs['TMA_ID'] == tma_id].copy()
    
    # qc metrics
    sc.pp.calculate_qc_metrics(adata, percent_top=(10, 20, 50, 150), inplace=True)
    
    df = pd.DataFrame({
        "TMA_ID": adata.obs["TMA_ID"],
        "Total_Transcripts": adata.obs["total_counts"],
        "Unique_Transcripts": adata.obs["n_genes_by_counts"],
        "Cell_Area": adata.obs["cell_area"],
        "Nucleus_Area": adata.obs["nucleus_area"]
    })
    qc_data.append(df)

qc_df = pd.concat(qc_data)

# plotting for metrics and transcript

with PdfPages("outputfile.pdf") as pdf:

    for metric, title, ylabel in [
        ("Total_Transcripts", "Total Transcripts per Cell", "Counts"),
        ("Unique_Transcripts", "Unique Transcripts per Cell", "Counts"),
        ("Cell_Area", "Cell Area", "Area"),
        ("Nucleus_Area", "Nucleus Area", "Area"),
    ]:
        fig, ax = plt.subplots(figsize=(10, 6))
        sns.boxplot(data=qc_df, x="TMA_ID", y=metric, ax=ax)
        ax.set_title(title, fontsize=16, fontweight='bold')
        ax.set_ylabel(ylabel)
        ax.set_xlabel("TMA ID")
        ax.set_xticklabels(ax.get_xticklabels(), rotation=90)
        plt.tight_layout()
        pdf.savefig(fig)
        plt.close(fig)
